#### 2.1 Knapsack problem with rules: 
Rules are important for defining indexed constraints, however, they can also be used for single (i.e. scalar) constraints. Here we reimplement the model using
rules for the objective and the constraints.

In [ ]:
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys

if "google.colab" in sys.modules:
    required_solvers = {'cbc'}

    if importlib.util.find_spec("pyomo") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyomo"])

    apt_packages = []
    if "glpk" in required_solvers and shutil.which("glpsol") is None:
        apt_packages.append("glpk-utils")
    if "cbc" in required_solvers and shutil.which("cbc") is None:
        apt_packages.append("coinor-cbc")

    if apt_packages:
        subprocess.check_call(["apt-get", "update"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq", *apt_packages])

    if "ipopt" in required_solvers and shutil.which("ipopt") is None:
        if importlib.util.find_spec("idaes") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "idaes-pse"])
        idaes_bin = Path.home() / ".idaes" / "bin"
        subprocess.check_call(["idaes", "get-extensions"])
        os.environ["PATH"] = f"{idaes_bin}:{os.environ['PATH']}"

    missing_solvers = []
    if "glpk" in required_solvers and shutil.which("glpsol") is None:
        missing_solvers.append("glpk")
    if "cbc" in required_solvers and shutil.which("cbc") is None:
        missing_solvers.append("cbc")
    if "ipopt" in required_solvers and shutil.which("ipopt") is None:
        missing_solvers.append("ipopt")

    if missing_solvers:
        raise RuntimeError(
            "Missing solver executables after Colab setup: "
            + ", ".join(sorted(missing_solvers))
        )


In [1]:
import pyomo.environ as pyo

A = ['hammer', 'wrench', 'screwdriver', 'towel']
b = {'hammer':8, 'wrench':3, 'screwdriver':6, 'towel':11}
w = {'hammer':5, 'wrench':7, 'screwdriver':4, 'towel':3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var( A, within=pyo.Binary )

def obj_rule(m):
    return sum( b[i]*m.x[i] for i in A )
model.obj = pyo.Objective(rule=obj_rule, sense = pyo.maximize )

def weight_con_rule(m):
    return sum( w[i]*m.x[i] for i in A ) <= W_max
model.weight_con = pyo.Constraint(rule=weight_con_rule)

opt = pyo.SolverFactory('cbc')
opt_success = opt.solve(model)

model.pprint()

1 Var Declarations
    x : Size=4, Index={hammer, wrench, screwdriver, towel}
        Key         : Lower : Value : Upper : Fixed : Stale : Domain
             hammer :     0 :   1.0 :     1 : False : False : Binary
        screwdriver :     0 :   1.0 :     1 : False : False : Binary
              towel :     0 :   1.0 :     1 : False : False : Binary
             wrench :     0 :   0.0 :     1 : False : False : Binary

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 8*x[hammer] + 3*x[wrench] + 6*x[screwdriver] + 11*x[towel]

1 Constraint Declarations
    weight_con : Size=1, Index=None, Active=True
        Key  : Lower : Body                                                      : Upper : Active
        None :  -Inf : 5*x[hammer] + 7*x[wrench] + 4*x[screwdriver] + 3*x[towel] :  14.0 :   True

3 Declarations: x obj weight_con
